In [5]:
import numpy as np
import pickle
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
#from knock72 import BoWClassifier  # We will inline BoWClassifier
#from knock75 import collate        # We'll inline collate

# inline BoWClassifier and collate definitions here for self-contained execution
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence
import pickle

# データロード
with open('70_embeddings.npy', 'rb') as f:
    embedding_matrix = np.load(f)
with open('71_sst_data.pkl', 'rb') as f:
    data = pickle.load(f)
train_data = data['train']
dev_data = data['dev']

# collate
from torch.nn.utils.rnn import pad_sequence
def collate(batch):
    batch_sorted = sorted(batch, key=lambda x: x['input_ids'].size(0), reverse=True)
    inputs = [ex['input_ids'] for ex in batch_sorted]
    labels = [ex['label'] for ex in batch_sorted]
    input_ids_padded = pad_sequence(inputs, batch_first=True, padding_value=0)
    labels_tensor = torch.cat(labels).view(-1,1)
    return {'input_ids': input_ids_padded, 'label': labels_tensor}

# BoWClassifier
class BoWClassifier(nn.Module):
    def __init__(self, embedding_matrix):
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(
            torch.tensor(embedding_matrix, dtype=torch.float), freeze=False, padding_idx=0)
        emb_dim = embedding_matrix.shape[1]
        self.weight = nn.Parameter(torch.zeros(emb_dim))
        self.bias = nn.Parameter(torch.zeros(1))
    def forward(self, input_ids):
        emb = self.embedding(input_ids)
        mask = (input_ids != 0).unsqueeze(-1).float()
        summed = (emb * mask).sum(dim=1)
        lengths = mask.sum(dim=1).clamp(min=1)
        avg = summed / lengths
        logits = avg.matmul(self.weight) + self.bias
        return torch.sigmoid(logits)

model = BoWClassifier(embedding_matrix)
optimizer = torch.optim.Adam([model.weight, model.bias], lr=1e-3)
criterion = nn.BCELoss()

dataloader = DataLoader(train_data, batch_size=32, shuffle=True, collate_fn=collate)

# 訓練
model.train()
for epoch in range(3):
    for batch in dataloader:
        x = batch['input_ids']
        y = batch['label'].squeeze()
        optimizer.zero_grad()
        pred = model(x)
        loss = criterion(pred, y)
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch} completed.")

# 評価
model.eval()
correct = 0
for i in range(0, len(dev_data), 32):
    batch = collate(dev_data[i:i+32])
    x = batch['input_ids']
    y = batch['label'].squeeze()
    correct += (model(x).round() == y).sum().item()
acc = correct / len(dev_data)
print(f"Dev accuracy: {acc:.4f}")

Epoch 0 completed.
Epoch 1 completed.
Epoch 2 completed.
Dev accuracy: 0.7833
